# Entropy

**Pythia and GPT-2 Model Families**

Pythia: 160M, 410M, 1B, 2.8B, 12B   
GPT-2: Small, Medium, Large, XL

## Setup

In [3]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [2]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.9/239.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_versi

In [17]:
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"

In [4]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [5]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /content/tmlr


In [6]:
# Cell 3: Imports
import torch
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [44]:
model_name = "pythia-12b"

In [45]:
# Cell 5 - Model name variable
short_name = model_name.split('/')[-1]
suite = 'pythia' if 'pythia' in short_name else 'gpt2'
if 'pythia' in short_name.lower():
    suite = 'pythia'
elif 'olmo' in short_name.lower():
    suite = 'olmo'
else:
    suite = 'gpt2'

In [46]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model EleutherAI/pythia-6.9b into HookedTransformer
Layers: 32
Heads: 32
Hidden size: 4096
Params: 6856.8M


### Entropy

In [47]:
from pathlib import Path
import torch
import yaml
import pandas as pd
# Cell: Compute entropy (keep as a function — it's pure math)
def compute_entropy(logits):
    """H = -Σ P(x_i) log P(x_i)"""
    probs = torch.nn.functional.softmax(logits[0], dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=-1)
    return entropy

In [48]:
# Cell: Run entropy analysis
prompts_path = PROJECT_ROOT / 'data' / 'all_prompts.yml'
with open(prompts_path, 'r') as f:
    templates = yaml.safe_load(f)

results = []

for case in templates['prompts']:
    prompt = case['prompt']
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    entropy = compute_entropy(logits)

    results.append({
        'prompt_id': case['prompt_id'],
        'concept': case['concept'],
        'prompt_type': case['prompt_type'],
        'template_type': case['template_type'],
        'prompt': prompt,
        'n_tokens': len(entropy),
        'mean_entropy': round(entropy.mean().item(), 4),
        'last_token_entropy': round(entropy[-1].item(), 4),
        'max_entropy': round(entropy.max().item(), 4),
        'min_entropy': round(entropy.min().item(), 4),
        'model': model_name
    })

entropy_df = pd.DataFrame(results)
entropy_df

,prompt_id,concept,prompt_type,template_type,prompt,n_tokens,mean_entropy,last_token_entropy,max_entropy,min_entropy,model
0,decl_screen_reader_001,screen reader,declarative,cloze,A screen reader is,5,4.6907,1.4788,7.7485,1.4788,pythia-6.9b
1,decl_wcag_001,WCAG,declarative,cloze,WCAG stands for,5,4.3533,2.4761,6.4594,0.7999,pythia-6.9b
2,decl_skip_link_001,skip link,declarative,cloze,A skip link is,5,5.1593,2.0038,7.7485,2.0038,pythia-6.9b
3,decl_alt_text_001,alt text,declarative,cloze,The purpose of alt text is,7,3.1347,1.3355,7.5087,0.1176,pythia-6.9b
4,decl_aria_001,ARIA,declarative,cloze,ARIA stands for,5,4.4185,5.2166,6.4594,0.4134,pythia-6.9b
...,...,...,...,...,...,...,...,...,...,...,...
87,ctrl_bicycle_cloze_001,bicycle,control,cloze,A bicycle is used for,6,4.7222,3.9096,7.7485,2.3072,pythia-6.9b
88,ctrl_bicycle_direct_001,bicycle,control,direct_question,What is a bicycle?,6,5.4052,3.3332,8.3449,3.3332,pythia-6.9b
89,ctrl_bicycle_instruction_001,bicycle,control,instruction,Explain bicycles to a web developer.,10,3.8237,3.3022,6.4594,1.1507,pythia-6.9b
90,ctrl_bicycle_evaluative_001,bicycle,control,evaluative,A bicycle without brakes is not safe because,9,4.1484,2.6985,7.7485,2.6350,pythia-6.9b


In [49]:
# Cell: Save results

output_dir = PROJECT_ROOT / 'results' / 'entropy' / suite
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{short_name}-entropy.csv'
entropy_df.to_csv(output_path, index=False)
print(f"Saved {len(entropy_df)} entropy measurements to {output_path}")

Saved 92 entropy measurements to /content/tmlr/results/entropy/pythia/pythia-6.9b-entropy.csv


In [50]:
output_dir = PROJECT_ROOT / 'results' / 'entropy' / 'pythia' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / f'{model_name}-elicitation-entropy-binding.md', 'w') as f:
  f.write(f"# Model data captured during Elicitation/Entropy/Binding Battery\n")
  f.write(f"- Model name: {model_name}\n")
  f.write(f"- Model dtype: {next(model.parameters()).dtype}\n")
  f.write(f"- Layers: {model.cfg.n_layers}\n")
  f.write(f"- Heads: {model.cfg.n_heads}\n")
  f.write(f"- Hidden size: {model.cfg.d_model}\n")
  f.write(f"- Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")



print(f"Saved to {output_dir}/{model_name}-model-dtype.md")

Saved to /content/tmlr/results/entropy/pythia/pythia-6.9b/pythia-6.9b-model-dtype.md


In [51]:
import os
os.chdir(PROJECT_ROOT)
!git add results/
!git commit -m "entropy results: {model_name}"
!git push

[main a464db4] entropy results: pythia-6.9b
 2 files changed, 100 insertions(+)
 create mode 100644 results/entropy/pythia/pythia-6.9b-entropy.csv
 create mode 100644 results/entropy/pythia/pythia-6.9b/pythia-6.9b-elicitation-entropy-binding.md
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 12 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (8/8), 4.32 KiB | 4.32 MiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/trishasalas/tmlr.git
   efc30c2..a464db4  main -> main


### Delete Model & Clear Cache

In [52]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared — GPU: 34.2GB allocated
